# PPM rewrites

Build and display the two motivating rewrites, then verify their full quantum channels. First-use resets prepare ancillas; terminal Z readouts consume wires. These Pauli corrections require no branch enumeration.

In [1]:
import sys
from pathlib import Path

# Works from the repository root or this notebook's directory.
root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "resource_estimation" / "clifford_verification").is_dir()
)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import cirq

from resource_estimation.clifford_verification import cliffords_equal

## CNOT → XXX measurement

The control moves from `a_in` to fresh `a_out`; `b` remains the target.

In [2]:
a_in = cirq.NamedQubit("a_in")
a_out = cirq.NamedQubit("a_out")
b = cirq.NamedQubit("b")
order = [a_in, a_out, b]

original = cirq.Circuit(cirq.CNOT(a_in, b))
xxx = cirq.measure_single_paulistring(cirq.X(a_in) * cirq.X(a_out) * cirq.X(b), key="p")
z_readout = cirq.measure(a_in, key="m")
body = cirq.Circuit(
    xxx,
    z_readout,
    cirq.Z(a_out).with_classical_controls("p"),
    cirq.X(a_out).with_classical_controls("m"),
    cirq.X(b).with_classical_controls("m"),
)

optimized = cirq.Circuit(cirq.reset(a_out)) + body

print("Original CNOT:")
print(original)
print("\nPPM replacement:")
print(optimized.to_text_diagram(qubit_order=order))

Original CNOT:
a_in: ───@───
         │
b: ──────X───

PPM replacement:
                     ┌──┐   ┌──┐
a_in: ────────M(X)────M────────────
              ║       ║
a_out: ───R───M(X)────╫Z─────X─────
              ║       ║║     ║
b: ───────────M(X)────╫╫─────╫X────
              ║       ║║     ║║
m: ═══════════╬═══════@╬═════^^════
              ║        ║
p: ═══════════@════════^═══════════
                     └──┘   └──┘


In [3]:
assert cliffords_equal(original, optimized)
uncorrected = cirq.Circuit(cirq.reset(a_out), xxx, z_readout)
assert not cliffords_equal(original, uncorrected)
print("Verified: CNOT rewrite; removing corrections changes the channel.")

Verified: CNOT rewrite; removing corrections changes the channel.


## GHZ interactions → XXX and XXXX measurements

The resource is initially an arbitrary input. Fresh `a_out` and `b_out` replace the measured input wires; the resource is also measured.

In [4]:
import sympy

ha_in, ha_out, hb_in, hb_out, resource = map(
    cirq.NamedQubit, ["a_in", "a_out", "b_in", "b_out", "resource"]
)

ghz_original = cirq.Circuit(
    cirq.CNOT(ha_in, hb_in),
    cirq.CNOT(hb_in, resource),
    cirq.CNOT(ha_in, hb_in),
    cirq.measure(resource, key="s"),
)
ghz_uncorrected = cirq.Circuit(
    cirq.reset(ha_out),
    cirq.reset(hb_out),
    cirq.measure_single_paulistring(cirq.X(hb_in) * cirq.X(hb_out) * cirq.X(resource), key="q"),
    cirq.measure_single_paulistring(
        cirq.X(ha_in) * cirq.X(ha_out) * cirq.X(hb_in) * cirq.X(hb_out), key="p"
    ),
    cirq.measure(ha_in, key="m"),
    cirq.measure(hb_in, key="n"),
    cirq.measure(resource, key="l"),
)
ghz_corrections = cirq.Circuit(
    cirq.Z(ha_out).with_classical_controls(sympy.Xor(sympy.Symbol("p"), sympy.Symbol("q"))),
    cirq.Z(hb_out).with_classical_controls("q"),
    cirq.X(ha_out).with_classical_controls("m"),
    cirq.X(hb_out).with_classical_controls("n"),
)
ghz_optimized = ghz_uncorrected + ghz_corrections

print("Original GHZ circuit:")
print(ghz_original)
print("\nPPM replacement:")
print(ghz_optimized.to_text_diagram(qubit_order=[ha_in, ha_out, hb_in, hb_out, resource]))

Original GHZ circuit:
a_in: ───────@───────@────────
             │       │
b_in: ───────X───@───X────────
                 │
resource: ───────X───M('s')───

PPM replacement:
                        ┌──────────┐   ┌──┐   ┌──────────────────────┐   ┌──┐
a_in: ───────────────────M(X)───────────M───────────────────────────────────────
                         ║              ║
a_out: ──────R───────────M(X)───────────╫──────Z(conditions=[p ^ q])──────X─────
                         ║              ║      ║                          ║
b_in: ───────────M(X)────M(X)───────────╫M─────╫──────────────────────────╫─────
                 ║       ║              ║║     ║                          ║
b_out: ──────R───M(X)────M(X)───────────╫╫─────╫────────────────────Z─────╫X────
                 ║       ║              ║║     ║                    ║     ║║
resource: ───────M(X)────╫───M('l')─────╫╫─────╫────────────────────╫─────╫╫────
                 ║       ║              ║║     ║                    ║  

In [5]:
assert cliffords_equal(ghz_original, ghz_optimized)
assert not cliffords_equal(ghz_original, ghz_uncorrected)
print("Verified: GHZ rewrite; removing corrections changes the channel.")

Verified: GHZ rewrite; removing corrections changes the channel.


### Retain the classical result

Measurement records are normally discarded. Copy `s` and its recovered value `m XOR n XOR l` into an output wire to compare them too.

In [6]:
result_wire = cirq.NamedQubit("z_result")
ghz_original_with_result = (
    cirq.Circuit(cirq.reset(result_wire))
    + ghz_original
    + cirq.Circuit(cirq.X(result_wire).with_classical_controls("s"))
)
recovered_result = sympy.Xor(sympy.Symbol("m"), sympy.Symbol("n"), sympy.Symbol("l"))
ghz_optimized_with_result = (
    cirq.Circuit(cirq.reset(result_wire))
    + ghz_optimized
    + cirq.Circuit(cirq.X(result_wire).with_classical_controls(recovered_result))
)

assert cliffords_equal(ghz_original_with_result, ghz_optimized_with_result)
wrong_result = (
    cirq.Circuit(cirq.reset(result_wire))
    + ghz_optimized
    + cirq.Circuit(
        cirq.X(result_wire).with_classical_controls(sympy.Xor(sympy.Symbol("m"), sympy.Symbol("n")))
    )
)
assert not cliffords_equal(ghz_original_with_result, wrong_result)
print("Verified: retained readout s = m XOR n XOR l; missing l is detected.")

Verified: retained readout s = m XOR n XOR l; missing l is detected.


### Supply a T-state resource

Prepend the same compiler preparation declaration on both sides. See [Quick start](quick_start.ipynb) for the marker semantics.

In [7]:
from resource_estimation.compile_gateset.ppm_compiler import ResourceStateTag

t_marker = cirq.I(resource).with_tags(ResourceStateTag(cirq.T, 0, 0, 1))

prepared_original = cirq.Circuit(t_marker) + ghz_original_with_result
prepared_optimized = cirq.Circuit(t_marker) + ghz_optimized_with_result
assert cliffords_equal(prepared_original, prepared_optimized)
print("Verified with a tagged T-state resource.")

Verified with a tagged T-state resource.
